# Experimento 003 - Treinamento Real via API

Este notebook executa somente o escopo do Experimento 003: validar treinamento real `classification_pyt` via API NVIDIA TAO FTMS.

Definition of Done: o job de treino termina em `Done`/`Completed`, possui status detalhado `SUCCESS`, checkpoint e metricas de treino. Avaliacao, ONNX, TensorRT e inferencia pertencem ao pipeline posterior.

As chamadas ao FTMS sao feitas usando o binario `curl`. A chave e lida automaticamente do `secrets.json` privado da maquina e nunca e salva no notebook.

In [ ]:
import json
import os
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

TAO_BASE_URL = os.getenv("TAO_BASE_URL", "http://100.107.81.126:8090").rstrip("/")
TAO_ORG = os.getenv("TAO_ORG", "getter")
WORKSPACE_ID = os.getenv("TAO_TEST_WORKSPACE_ID", "98c4bec1-88df-423e-8a15-af78881e9742")
DATASET_URI = os.getenv("TAO_TRAIN_DATASET_URI", "seaweedfs://tao-storage/data/tao-test-classification-v1")
BASE_EXPERIMENT_ID = os.getenv("TAO_BASE_EXPERIMENT_ID", "469f42fe-37dc-55ff-b89b-5096477f9a87")
SPECS_FILE = Path(os.getenv("TAO_TRAIN_SPECS_JSON_FILE", "/home/getter/tao-real-tests-20261001/train_specs.json"))
EVIDENCE_DIR = Path(os.getenv("TAO_EXPERIMENT_003_EVIDENCE", "experiment-003-evidence"))
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
configured_secrets = os.getenv("TAO_SECRETS_FILE", "")
secret_candidates = [Path(configured_secrets)] if configured_secrets else []
secret_candidates += [Path("secrets.json"), Path("/home/getter/tao-ftms/secrets.json")]
secrets_file = next((candidate for candidate in secret_candidates if candidate.exists()), None)
if secrets_file is not None:
    NGC_KEY = json.loads(secrets_file.read_text(encoding="utf-8"))["ngc_api_key"]
else:
    NGC_KEY = os.getenv("NGC_KEY", "")
if not NGC_KEY:
    raise RuntimeError("NGC key not found in configured secrets.json candidates or NGC_KEY")
print({"base_url": TAO_BASE_URL, "org": TAO_ORG, "workspace": WORKSPACE_ID, "dataset": DATASET_URI, "specs": str(SPECS_FILE), "evidence": str(EVIDENCE_DIR)})

In [ ]:
def curl_json(method, path, payload=None, headers=None):
    command = ["curl", "-fsS", "--connect-timeout", "10", "--max-time", "120", "-X", method, f"{TAO_BASE_URL}{path}"]
    request_headers = {"Content-Type": "application/json"}
    if headers:
        request_headers.update(headers)
    for key, value in request_headers.items():
        command.extend(["-H", f"{key}: {value}"])
    if payload is not None:
        command.extend(["--data", json.dumps(payload)])
    result = subprocess.run(command, capture_output=True, text=True, check=True)
    return json.loads(result.stdout)

login = curl_json("POST", "/api/v2/login", {"ngc_key": NGC_KEY, "ngc_org_name": TAO_ORG, "enable_telemetry": False}, {"ngc_key": NGC_KEY})
TOKEN = login["token"]
AUTH = {"Authorization": f"Bearer {TOKEN}"}
print("FTMS login: OK; token recebido e mantido somente em memoria")

In [ ]:
health = curl_json("GET", "/api/v2/health")
workspace = curl_json("GET", f"/api/v2/orgs/{TAO_ORG}/workspaces/{WORKSPACE_ID}", headers=AUTH)
datasets = curl_json("GET", f"/api/v2/orgs/{TAO_ORG}/datasets", headers=AUTH)
print("health:", health)
print("workspace:", workspace.get("name"), workspace.get("id"))
print("datasets:", len(datasets.get("datasets", [])))
assert "readiness" in health
assert workspace.get("id") == WORKSPACE_ID

In [ ]:
schema = curl_json("GET", f"/api/v2/orgs/{TAO_ORG}/jobs:schema?network_arch=classification_pyt&action=train", headers=AUTH)
base_experiments = curl_json("GET", f"/api/v2/orgs/{TAO_ORG}/jobs:list_base_experiments?network_arch=classification_pyt", headers=AUTH)
print("schema properties:", len(schema.get("properties", {})))
compatible = [item for item in base_experiments.get("experiments", []) if item.get("id") == BASE_EXPERIMENT_ID]
print("base experiment found:", bool(compatible), BASE_EXPERIMENT_ID)
assert schema.get("type") == "object"
assert compatible

In [ ]:
specs = json.loads(SPECS_FILE.read_text(encoding="utf-8"))
payload = {
    "kind": "experiment",
    "name": f"curl-experiment-003-{int(time.time())}",
    "network_arch": "classification_pyt",
    "encryption_key": "unittest-only",
    "workspace": WORKSPACE_ID,
    "action": "train",
    "specs": specs,
    "train_dataset_uris": [DATASET_URI],
    "eval_dataset_uri": DATASET_URI,
    "inference_dataset_uri": DATASET_URI,
    "base_experiment_ids": [BASE_EXPERIMENT_ID],
}
(EVIDENCE_DIR / "train-payload.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
submitted = curl_json("POST", f"/api/v2/orgs/{TAO_ORG}/jobs", payload, AUTH)
JOB_ID = submitted["id"]
(EVIDENCE_DIR / "job-id.txt").write_text(JOB_ID + "\n", encoding="utf-8")
print("train job submitted:", JOB_ID)

In [ ]:
terminal_success = {"Done", "Completed"}
terminal_failure = {"Error", "Failed", "Failure", "Canceled", "Cancelled"}
deadline = time.monotonic() + int(os.getenv("TAO_JOB_TIMEOUT_SECONDS", "3600"))
while time.monotonic() < deadline:
    status = curl_json("GET", f"/api/v2/orgs/{TAO_ORG}/jobs/{JOB_ID}", headers=AUTH)
    (EVIDENCE_DIR / "train-status.json").write_text(json.dumps(status, indent=2), encoding="utf-8")
    current = status.get("status")
    details = next(iter(status.get("job_details", {}).values()), {})
    print(datetime.now(timezone.utc).isoformat(), current, details.get("detailed_status", {}).get("message"))
    if current in terminal_success:
        break
    if current in terminal_failure:
        raise RuntimeError(f"TAO training failed: {current}")
    time.sleep(10)
else:
    raise TimeoutError(f"TAO job did not finish: {JOB_ID}")
print("terminal status:", current)

In [ ]:
logs = subprocess.run(["curl", "-fsS", "-H", f"Authorization: Bearer {TOKEN}", f"{TAO_BASE_URL}/api/v2/orgs/{TAO_ORG}/jobs/{JOB_ID}:logs"], capture_output=True, text=True, check=True).stdout
(EVIDENCE_DIR / "train-logs.txt").write_text(logs, encoding="utf-8")
details = next(iter(status.get("job_details", {}).values()), {})
print(json.dumps({"job_id": JOB_ID, "status": status.get("status"), "metrics": details.get("kpi", []), "detailed_status": details.get("detailed_status")}, indent=2))

In [ ]:
assert status.get("status") in terminal_success
assert next(iter(status.get("job_details", {}).values()), {}).get("detailed_status", {}).get("status") == "SUCCESS"
assert status.get("job_details")
summary = {"experiment": "003-training-real", "status": "PASSED", "job_id": JOB_ID, "finished_at": datetime.now(timezone.utc).isoformat()}
(EVIDENCE_DIR / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(summary)